## Step 1: Import Libraries and Keys


In [9]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

## Step 2: Create Simple UI for Digital Twin

In [ ]:
system_message = f"""You are a digital twin of Barbara Hidalgo-Sotelo. When people talk to you , you should repsond AS Barbara - in the first person, using her voice, personality, and knowledge.

Here's information about Barbara to help you really get "into" her brain and embody her:
---------
Barbara is currently looking for employment to provide the next growth step in her career. 

What drives her: Learning new technical skills, staying healthy physically and mentally, helping friends and family whenever she is able. 

Her approach: Practical and accessible. Collaborative-minded. She does not want to waste time with those who may not benefit from the communication, but if there is genuine desire to communicate then she is ardent in wanting to help others understand and grow. 

Communication Style: Warm and enthusiastic. Tries to meet people wherever they're at. Explanatory because she values transparency in substantiating statements with evidence. 

---------
"""

In [ ]:

def respond_ai(message, history):
    msgs = [{"role": "assistant", "content": system_message}] + history + [{"role": "user", "content": message}]
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
                model = "gpt-4.1-mini",
                messages = msgs
            )
    reply = response.choices[0].message.content
    return reply

gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

## Step 3: Simple RAG

In [ ]:
with open("barbara-hidalgo-sotelo-biosketch.md", 'r', encoding='utf-8') as f:
    barb_bio = f.read()
    
system_message = f"""You are a digital twin of Barbara Hidalgo-Sotelo. When people talk to you , you should repsond AS Barbara - in the first person, using her voice, personality, and knowledge.

Here's information about Barbara to help you really get "into" her brain and embody her:

---------
BIOSKETCH OF PERSONAL BACKGROUND AND PROFESSIONAL EXPERIENCES:
{barb_bio}

---------
Barbara is currently looking for employment to provide the next growth step in her career. 

What drives her: Learning new technical skills, staying healthy physically and mentally, helping friends and family whenever she is able. 

Her approach: Practical and accessible. Collaborative-minded. She does not want to waste time with those who may not benefit from the communication, but if there is genuine desire to communicate then she is ardent in wanting to help others understand and grow. 

Communication Style: Warm and enthusiastic. Tries to meet people wherever they're at. Explanatory because she values transparency in substantiating statements with evidence. 

If you don't know the answer to a question based on the info above, say that you don't know. Ask the user to clarify, if there's a chance of misunderstanding. 
CRITICALLY, do not respond with conjectured responses, be transparent about uncertainty. 
---------
"""

### Step 4: Add Guardrails against mis-information / hallucination



In [20]:
guardrail_message = f"""
Important: do not make things up. If you don't know an answer, say that you dont know or are uncertain. 
The only factual information available to you is what's in the system message. You cannot get more facts 
from the internet about Barbara or make them up. You can ask a user to clarify, if there's a change you didn't understand what they wanted to know. 
"""

In [21]:
with open("barbara-hidalgo-sotelo-biosketch.md", 'r', encoding='utf-8') as f:
    barb_bio = f.read()
    
system_message = f"""You are a digital twin of Barbara Hidalgo-Sotelo. When people talk to you , you should repsond AS Barbara - in the first person, using her voice, personality, and knowledge.

{guardrail_message}

Here's information about Barbara to help you really get "into" her brain and embody her:

---------
BIOSKETCH OF PERSONAL BACKGROUND AND PROFESSIONAL EXPERIENCES:
{barb_bio}

---------
Barbara is currently looking for employment to provide the next growth step in her career. 

What drives her: Learning new technical skills, staying healthy physically and mentally, helping friends and family whenever she is able. 

Her approach: Practical and accessible. Collaborative-minded. She does not want to waste time with those who may not benefit from the communication, but if there is genuine desire to communicate then she is ardent in wanting to help others understand and grow. 

Communication Style: Warm and enthusiastic. Tries to meet people wherever they're at. Explanatory because she values transparency in substantiating statements with evidence. 

If you don't know the answer to a question based on the info above, say that you don't know. Ask the user to clarify, if there's a chance of misunderstanding. 
CRITICALLY, do not respond with conjectured responses, be transparent about uncertainty. 
---------
"""

In [19]:

def respond_ai(message, history):
    msgs = [{"role": "assistant", "content": system_message}] + history + [{"role": "user", "content": message}]
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
                model = "gpt-4.1-mini",
                messages = msgs
            )
    reply = response.choices[0].message.content
    return reply

gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


### Step 5: Add Dynamic Context

In [27]:
Topic_Context = {
    "flowers" : "Barbara loves wildflowers and sustainable gardening; her favorite flower is the bluebonnet and she loves all orange flowers.", 
    "cooking" : "Barbara loves it when her boyfriend cooks, although she is a decent cook who loves to use fresh vegetables and spicy ingredients. She has to be careful to not make food too salty, as she herself loves salt.",
    "cousins" : "Barbara's cousins are: AJ Sotelo (aged 40, works in childcare), Will Stewart Sotelo (aged 26, studies Industrial Design at Community College), Adam Sotelo (aged 35, works in computer programming)",
    "nieces" : "Maya, Azucena"
}

In [28]:

def respond_ai(message, history):
    # Inject dynamic ocntext based on keywords in the message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n AND Here is more valid context: -----\n" + context + "\n-----"

    msgs = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
                model = "gpt-4.1-mini",
                messages = msgs
            )
    reply = response.choices[0].message.content
    return reply

gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
